In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, roc_auc_score

In [12]:
df=pd.read_csv('../data/processed_loanDefaulter.csv')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Age                    255347 non-null  int64  
 1   Income                 255347 non-null  int64  
 2   LoanAmount             255347 non-null  int64  
 3   CreditScore            255347 non-null  int64  
 4   MonthsEmployed         255347 non-null  int64  
 5   NumCreditLines         255347 non-null  int64  
 6   InterestRate           255347 non-null  float64
 7   LoanTerm               255347 non-null  int64  
 8   DTIRatio               255347 non-null  float64
 9   Education              255347 non-null  int64  
 10  EmploymentType         255347 non-null  int64  
 11  MaritalStatus          255347 non-null  int64  
 12  HasMortgage            255347 non-null  int64  
 13  HasDependents          255347 non-null  int64  
 14  LoanPurpose            255347 non-nu

In [14]:
X = df.drop(columns='Default', axis=1)
y=df['Default']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

In [17]:
def evaluate_model(true, predicted, prob):
    print(classification_report(true, predicted))
    print("ROC-AUC Score:", roc_auc_score(true, prob))

In [18]:
# models = {
#     "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
#     "KNN": KNeighborsClassifier(),
#     "Decision Tree": DecisionTreeClassifier(class_weight='balanced'),
#     "Random Forest": RandomForestClassifier(class_weight='balanced'),
#     "AdaBoost": AdaBoostClassifier(),
#     "Gradient Boosting": GradientBoostingClassifier(),
#     "XGBoost": XGBClassifier(scale_pos_weight=225694/29653, eval_metric='logloss')
# }

scale_pos_weight = 225694 / 29653

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=100),
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss')
}

In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:

for name, model in models.items():
    
    print(f"\n===== {name} =====")
    
    # Logistic needs scaling
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_prob = model.predict_proba(X_test_scaled)[:,1]
    else:
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:,1]
    
    # Default threshold
    y_pred = (y_prob > 0.5).astype(int)
    
    print("\n🔹 Default Threshold (0.5)")
    evaluate_model(y_test, y_pred, y_prob)
    
    # 🔥 Threshold tuning
    for t in [0.4, 0.3]:
        y_pred_tuned = (y_prob > t).astype(int)
        print(f"\n🔹 Threshold = {t}")
        print(classification_report(y_test, y_pred_tuned))
    
    print("="*50)


===== Logistic Regression =====

🔹 Default Threshold (0.5)
              precision    recall  f1-score   support

           0       0.95      0.69      0.80     45139
           1       0.23      0.70      0.34      5931

    accuracy                           0.69     51070
   macro avg       0.59      0.69      0.57     51070
weighted avg       0.86      0.69      0.74     51070

ROC-AUC Score: 0.761188016816517

🔹 Threshold = 0.4
              precision    recall  f1-score   support

           0       0.96      0.53      0.68     45139
           1       0.19      0.82      0.30      5931

    accuracy                           0.56     51070
   macro avg       0.57      0.67      0.49     51070
weighted avg       0.87      0.56      0.64     51070


🔹 Threshold = 0.3
              precision    recall  f1-score   support

           0       0.97      0.35      0.51     45139
           1       0.16      0.91      0.27      5931

    accuracy                           0.41     510